In [13]:
import numpy as np
import pandas as pd
import xgboost as xgb
import json

In [5]:
data = pd.read_parquet(r"../Data/sale_and_crime_data/chicago_sales_with_crime.parquet")

In [14]:
features_to_exclude = ["pin","is_multisale",'flood_fema_sfha', "sale_filter_same_sale_within_365", "sale_filter_less_than_10k", "single_v_multi_family", "zip_code",
                      "longitude", "latitude", "sale_date_parsed", "class", 'centroid_x_crs_3435', 'centroid_y_crs_3435', 'pin10', 'nearest_metra_route_dist_ft', 'nearest_new_construction_pin10',
                       "school_elementary_district_name",'nearest_golf_course_dist_ft', "school_secondary_district_name", "township_code",  "neighborhood_code",'sale_filter_deed_type', 'nearest_cta_route_dist_ft',
                                 'nearest_vacant_land_pin10','airport_noise_dnl','attic_finish','census_acs5_tract_geoid', 'nearest_neighbor_1_pin10','nearest_neighbor_2_pin10','nearest_neighbor_3_pin10','sale_date_parsed','sale_date', 'sale_year']
data = data.drop(columns = features_to_exclude)

In [15]:
cat_cols = ["type_of_residence", "construction_quality", "garage_size",
            "basement_type", "ext_wall_material", "repair_condition", "basement_finish",
            "central_heating", "central_air", "community_area"]

for col in cat_cols:
    data[col] = data[col].astype("category")

In [16]:
def XGBModel(df, hyperparams):
    y=df["sale_price"]
    features = list(df.columns)
    features.remove("sale_price")
    X=df[features]
    
    model = xgb.XGBRegressor(
        random_state=42, 
        enable_categorical=True, 
        tree_method="hist",
        **hyperparams
    )
    model.fit(
        X,y,
        verbose=False
    )
    return model

In [17]:
data_first = data[data["year"]<2023]

In [18]:
data_second = data[data["year"]>=2023]

In [20]:
with open(r"..\notebooks\hyperparameters.json", "r") as f:
    hyperparams = json.load(f)

In [21]:
model_2020_2023 = XGBModel(data_first, hyperparams)
model_2023_present = XGBModel(data_second, hyperparams)

In [22]:
def categorical_pdp(model, X, feature, categories=None):
    """
    Compute partial dependence (marginal expected value) for a categorical feature.
    
    model: fitted XGBRegressor (enable_categorical=True)
    X: DataFrame used for prediction (same columns/dtypes as training)
    feature: name of the categorical column
    categories: optional list/order of categories to evaluate; defaults to all observed categories
    """
    if categories is None:
        categories = X[feature].cat.categories if hasattr(X[feature], "cat") else X[feature].unique()

    results = []
    X_temp = X.copy()

    for cat in categories:
        X_temp[feature] = cat
        # preserve categorical dtype/categories so xgboost doesn't complain
        X_temp[feature] = X_temp[feature].astype(X[feature].dtype)
        preds = model.predict(X_temp)
        results.append({"category": cat, "mean_prediction": preds.mean(), "std_prediction": preds.std()})

    return pd.DataFrame(results).sort_values("mean_prediction", ascending=False)

In [33]:
X1=data_first.drop(columns="sale_price")
X2=data_second.drop(columns="sale_price")

In [34]:
Change_in_pdp = categorical_pdp(model_2023_present, X2, "community_area")-categorical_pdp(model_2020_2023, X1, "community_area")

In [35]:
print(Change_in_pdp.to_string(index=False))

 category  mean_prediction  std_prediction
        0     73294.125000    36216.953125
        0     27369.312500    21110.593750
        0     52957.250000    32619.718750
        0     36579.937500    66532.093750
        0    128391.312500   101982.062500
        0     45066.625000    98888.156250
        0     85028.500000   117262.656250
        0    123325.125000    89976.531250
        0     31191.406250    25902.781250
        0     26027.218750    34688.765625
        0     20459.093750    22367.656250
        0     24397.281250    43604.031250
        0     40827.062500    25663.109375
        0     38436.468750    48601.812500
        0     27586.687500    31141.140625
        0     28007.125000    39348.500000
        0     26362.343750    31069.671875
        0     29812.406250    35853.109375
        0     33785.031250    30253.265625
        0      3616.000000    26038.593750
        0    116958.093750    75786.062500
        0     45035.312500    88996.687500
        0  